In [ ]:
# NOTE: you CAN change this cell
# import your library here
import time
# from vietnam_provinces import ProvinceCode, Province, WardCode, Ward
import re
import itertools
import unicodedata
import difflib

In [ ]:
# import re
# import itertools
# import unicodedata
# import difflib

class Solution:
    def __init__(
        self,
        list_province: list[str],
        list_ward: list[str],
        list_street: list[str]
    ):
        self.list_province = list_province
        self.list_ward = list_ward
        self.list_street = list_street

        # O(1) Sets for instant exact-match verification (guarantees DB-only output)
        self.set_province = set(list_province)
        self.set_ward = set(list_ward)
        self.set_street = set(list_street)

        # Lowercase mapping for case-insensitive fuzzy matching
        self.prov_lower = {c.lower(): c for c in list_province}
        self.ward_lower = {c.lower(): c for c in list_ward}
        self.street_lower = {c.lower(): c for c in list_street}

        # Cache to prevent redundant fuzzy-search calculations and maintain 0.0001s speed
        self.fuzzy_cache = {}

        # Mappers to handle completely unspaced inputs
        self.re_prov_map = re.compile(r'^(?:thành phố|thanh pho|tỉnh|tinh|tp\.|t\.|tp\s+|t\s+|province|city)\s*', re.IGNORECASE)
        self.re_ward_map = re.compile(r'^(?:phường|phuong|xã|xa|thị trấn|thi tran|p\.|x\.|tt\.|p\s+|x\s+|tt\s+|ward)\s*', re.IGNORECASE)
        self.re_street_map = re.compile(r'^(?:(?:đường|duong|phố|pho|đại lộ|dai lo|khu dân cư|khu dan cu|ngõ|ngo|ngách|ngach|hẻm|hem|kiệt|kiet|số|so)\s+)+', re.IGNORECASE)

        # Aggressive stripper exclusively used for generating Fallback strings
        self.re_street_aggro = re.compile(r'^(?:(?:ngách|ngõ|hẻm|kiệt|số|đc)\s*[\d/.-]+[a-zA-Z]*\s*)*(?:[\d/.-]+[a-zA-Z]*\s+)?(?:đường\s+|phố\s+|đại lộ\s+|khu dân cư\s+)?', re.IGNORECASE)

        # Build Dual-Tier Maps (Accented and Unaccented)
        self.prov_acc, self.prov_unacc = self._build_tiered_maps(list_province, self.re_prov_map)
        self.ward_acc, self.ward_unacc = self._build_tiered_maps(list_ward, self.re_ward_map)
        self.street_acc, self.street_unacc = self._build_tiered_maps(list_street, self.re_street_map)

    def _normalize_vietnamese(self, s: str) -> str:
        if not s: return s
        return unicodedata.normalize('NFC', s)

    def _unaccent(self, s: str) -> str:
        if not s: return ""
        s = s.replace('đ', 'd').replace('Đ', 'D')
        return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

    def _dense(self, s: str) -> str:
        if not s: return ""
        return self._normalize_vietnamese(s).replace(" ", "").lower()

    def _fix_typos(self, s: str) -> str:
        s = s.strip()
        # Fix missing spacing on single-letter abbreviations
        if len(s) > 1 and s[0].lower() in ('p', 't', 'x') and s[1].isupper():
            s = s[0].upper() + '. ' + s[1:]
        elif len(s) > 2 and s[:2].lower() in ('tp', 'tt') and s[2].isupper():
            s = s[:2].upper() + '. ' + s[2:]

        # Regex boundary replacement catches typos anywhere in the string (e.g. "210 ặng" -> "210 Đặng")
        s = re.sub(r'(^|\s)[oO]àn(\s|$)', r'\1Đoàn\2', s)
        s = re.sub(r'(^|\s)[ặẶ]ng(\s|$)', r'\1Đặng\2', s)
        s = re.sub(r'(^|\s)[ạẠ]i(\s|$)', r'\1Đại\2', s)
        s = re.sub(r'(^|\s)[đĐ]ng(\s|$)', r'\1Đặng\2', s)

        return s

    def _build_tiered_maps(self, string_list: list[str], regex_pattern: re.Pattern):
        acc_map, unacc_map = {}, {}
        for item in string_list:
            norm_item = self._normalize_vietnamese(item)
            raw_acc = self._dense(norm_item)
            clean_item = regex_pattern.sub('', norm_item).strip()
            clean_acc = self._dense(clean_item)

            if raw_acc not in acc_map: acc_map[raw_acc] = item
            if clean_acc and clean_acc not in acc_map: acc_map[clean_acc] = item

            raw_unacc, clean_unacc = self._unaccent(raw_acc), self._unaccent(clean_acc)
            if raw_unacc and raw_unacc not in unacc_map: unacc_map[raw_unacc] = item
            if clean_unacc and clean_unacc not in unacc_map: unacc_map[clean_unacc] = item

        return acc_map, unacc_map

    def _suffix_match(self, dense_str: str, target_map: dict):
        for i in range(len(dense_str) - 1):
            suffix = dense_str[i:]
            if suffix in target_map:
                return target_map[suffix]
        return None

    def _has_prefix(self, s: str, regex_pattern: re.Pattern) -> bool:
        m = regex_pattern.match(s)
        return m is not None and m.end() > 0

    def _format_fallback(self, s: str) -> str:
        if not s: return ""
        return " ".join(w.capitalize() for w in s.split())

    def _score_assignment(self, part: str, slot: str):
        if not part: return -100, ""

        fixed_part = self._fix_typos(part)
        norm_part = self._normalize_vietnamese(fixed_part)

        if slot == 'province':
            re_map, re_aggro = self.re_prov_map, self.re_prov_map
            acc_map, unacc_map = self.prov_acc, self.prov_unacc
        elif slot == 'ward':
            re_map, re_aggro = self.re_ward_map, self.re_ward_map
            acc_map, unacc_map = self.ward_acc, self.ward_unacc
        else:
            re_map, re_aggro = self.re_street_map, self.re_street_aggro
            acc_map, unacc_map = self.street_acc, self.street_unacc

        c_part = re_map.sub('', norm_part).strip()

        u_raw_acc, u_clean_acc = self._dense(norm_part), self._dense(c_part)
        u_raw_unacc, u_clean_unacc = self._unaccent(u_raw_acc), self._unaccent(u_clean_acc)

        matched, val = False, ""

        if u_raw_acc in acc_map: matched, val = True, acc_map[u_raw_acc]
        elif u_clean_acc in acc_map: matched, val = True, acc_map[u_clean_acc]
        else:
            match_acc = self._suffix_match(u_raw_acc, acc_map)
            if match_acc: matched, val = True, match_acc
            else:
                if u_raw_unacc in unacc_map: matched, val = True, unacc_map[u_raw_unacc]
                elif u_clean_unacc in unacc_map: matched, val = True, unacc_map[u_clean_unacc]
                else:
                    match_unacc = self._suffix_match(u_raw_unacc, unacc_map)
                    if match_unacc: matched, val = True, match_unacc

        score = 0
        if matched: score += 100
        else: val = self._format_fallback(re_aggro.sub('', norm_part).strip())

        if self._has_prefix(norm_part, re_map): score += 50

        if slot != 'province' and (u_raw_acc in self.prov_acc or u_raw_unacc in self.prov_unacc): score -= 50
        if slot != 'ward' and (u_raw_acc in self.ward_acc or u_raw_unacc in self.ward_unacc): score -= 50
        if slot != 'street' and (u_raw_acc in self.street_acc or u_raw_unacc in self.street_unacc): score -= 50

        return score, val

    def _closest_match(self, val: str, lower_map: dict, exact_set: set) -> str:
        """Case-insensitive fuzzy matcher strictly enforces database outputs"""
        if not val:
            return ""

        # 1. Skip if perfectly matched already
        if val in exact_set:
            return val

        # 2. Check cache
        if val in self.fuzzy_cache:
            return self.fuzzy_cache[val]

        # 3. Case-Insensitive Fuzzy Matching (neutralizes 'Ặ' vs 'ặ' unicode penalty)
        val_lower = val.lower()
        matches = difflib.get_close_matches(val_lower, lower_map.keys(), n=1, cutoff=0.85)

        if matches:
            res = lower_map[matches[0]]
            self.fuzzy_cache[val] = res
            return res

        # If no strict match is found, enforce empty string
        self.fuzzy_cache[val] = ""
        return ""

    def process(self, s: str):
        parts = [p.strip() for p in s.split(',') if p.strip()]

        if len(parts) > 3: parts = [", ".join(parts[:-2]), parts[-2], parts[-1]]

        L = len(parts)
        if L == 0: return {"street": "", "ward": "", "province": ""}

        alignments = list(itertools.combinations([0, 1, 2], L))
        slot_names = ["street", "ward", "province"]
        tie_breakers = {0: 0.3, 1: 0.2, 2: 0.1}

        best_score, best_assignment = -float('inf'), {"street": "", "ward": "", "province": ""}

        for align in alignments:
            current_score, current_assignment = 0, {"street": "", "ward": "", "province": ""}

            for i, slot_idx in enumerate(align):
                slot_name = slot_names[slot_idx]
                score, val = self._score_assignment(parts[i], slot_name)

                current_score += score + tie_breakers[slot_idx]
                current_assignment[slot_name] = val

            if current_score > best_score:
                best_score, best_assignment = current_score, current_assignment

        # Final pass: Case-insensitive strict DB mappings
        best_assignment["province"] = self._closest_match(best_assignment["province"], self.prov_lower, self.set_province)
        best_assignment["ward"] = self._closest_match(best_assignment["ward"], self.ward_lower, self.set_ward)
        best_assignment["street"] = self._closest_match(best_assignment["street"], self.street_lower, self.set_street)

        return best_assignment

In [ ]:
# Download public test
!rm -rf test.json
!gdown --fuzzy https://drive.google.com/file/d/18qzaHmx9-fcKPi4gTq304G2JvXy_g6XW/view?usp=sharing -O test.json
!gdown --fuzzy https://drive.google.com/file/d/1PHRkRLRaO2vFivLbcPJsaxjvZ7K_I3lU/view?usp=drive_link -O list_province.txt
!gdown --fuzzy https://drive.google.com/file/d/1QrSouYn6IUEW7Impje1vOxzsksjlAJ-h/view?usp=drive_link -O list_ward.txt
!gdown --fuzzy https://drive.google.com/file/d/1yP2dBE8K9QtUiaLPewtQRr-hDH0gOgfa/view?usp=drive_link -O list_street.txt